# 05 — Latency Distribution

**V6**: p50/p95 of `submit_latency_us` and `fill_latency_us` from fills table.
Also shows `gate_latency_us` from risk decisions and `signal_to_submit_latency_us` from the attribution view.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
from src import db

In [ ]:
# Pull from live_trade_attribution view for full latency chain
rows = db.query("""
    SELECT
        correlation_id,
        gate_latency_us,
        submit_latency_us,
        fill_latency_us,
        signal_to_submit_latency_us,
        signal_to_fill_latency_us
    FROM live_trade_attribution
""")
lat = pd.DataFrame(rows)
print(f'Attribution rows: {len(lat)}')
lat.head(3)

In [ ]:
# V6 — p50/p95 per latency field
cols = ['gate_latency_us', 'submit_latency_us', 'fill_latency_us',
        'signal_to_submit_latency_us', 'signal_to_fill_latency_us']

summary = {}
for col in cols:
    s = pd.to_numeric(lat[col], errors='coerce').dropna()
    if len(s):
        summary[col] = {'p50': s.quantile(0.50), 'p95': s.quantile(0.95), 'n': len(s)}

df_summary = pd.DataFrame(summary).T
print(df_summary.to_string())

In [ ]:
# Distribution plot for signal_to_submit and fill latency
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, label in [
    (axes[0], 'signal_to_submit_latency_us', 'Signal→Submit (µs)'),
    (axes[1], 'fill_latency_us', 'Fill latency (µs)'),
]:
    s = pd.to_numeric(lat[col], errors='coerce').dropna()
    if len(s):
        ax.hist(s, bins=40, color='steelblue', edgecolor='white')
        ax.axvline(s.quantile(0.50), color='orange', linestyle='--', label=f'p50={s.quantile(.5):.0f}')
        ax.axvline(s.quantile(0.95), color='red',    linestyle='--', label=f'p95={s.quantile(.95):.0f}')
        ax.set_title(label)
        ax.legend()
    else:
        ax.set_title(f'{label} — no data')

plt.tight_layout()
plt.savefig('../data/raw/latency.png', dpi=120)
plt.show()